# 04 — Error Analysis: When Does Each Model Win?

`02_baselines.ipynb` and `03_ml_models.ipynb` established that SARIMA
(4.50% MAPE) narrowly beat LightGBM (4.96% MAPE) *on average*. But an
average hides structure — the interesting question isn't just "which model
is better overall," it's **when** and **why** one wins. This notebook
breaks the comparison down by season, time of day, and temperature to find
that structure, rather than adding another model.

**Setup required first:** this notebook loads saved predictions from the
earlier notebooks. Add this cell to the end of `02_baselines.ipynb` (after
the SARIMA backtest cell) and `03_ml_models.ipynb` (after the LightGBM
backtest cell) if you haven't already:

```python
# in 02_baselines.ipynb, after sarima_result is computed:
sarima_result.predictions.to_csv("../reports/sarima_predictions.csv", index=False)

# in 03_ml_models.ipynb, after lgb_result is computed:
lgb_result.predictions.to_csv("../reports/lightgbm_predictions.csv", index=False)
```

Re-run those two cells once to produce the CSVs, then run this notebook.


In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (12, 4)

def mape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    mask = y_true != 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)

sarima_preds = pd.read_csv("../reports/sarima_predictions.csv", parse_dates=["timestamp"])
lgb_preds = pd.read_csv("../reports/lightgbm_predictions.csv", parse_dates=["timestamp"])

# Merge on timestamp so every row has both models' predictions for the
# same true value, plus the original weather/calendar data for context.
raw = pd.read_csv("../data/processed/load_weather_hourly.csv", index_col=0, parse_dates=True)
raw.index.name = "timestamp"

merged = sarima_preds.rename(columns={"y_pred": "sarima_pred"})[["timestamp", "y_true", "sarima_pred"]].merge(
    lgb_preds.rename(columns={"y_pred": "lgb_pred"})[["timestamp", "lgb_pred"]],
    on="timestamp", how="inner"  # inner join: only compare timestamps both models actually forecasted
)
merged = merged.merge(raw[["temperature_2m", "is_holiday"]], left_on="timestamp", right_index=True, how="left")
merged["month"] = merged["timestamp"].dt.month
merged["hour"] = merged["timestamp"].dt.hour

print(f"{len(merged)} timestamps with both models' predictions available for comparison")
merged.head()

**Note on the join:** SARIMA and LightGBM were backtested on different
fold schedules in the earlier notebooks (different `step` sizes for
runtime reasons), so they don't necessarily share the exact same set of
forecasted timestamps. The inner join above only keeps timestamps both
models actually predicted, which is the fair, apples-to-apples subset for
this comparison — but it does mean this notebook's numbers may differ
slightly from the full backtest averages reported in the README, since
it's evaluated on a (large) common subset rather than each model's full
fold set. Worth stating this explicitly rather than presenting the numbers
as identical to the earlier ones.


## 1. Seasonal breakdown

Delhi has three broad seasons relevant to electricity demand: summer
(intense heat, AC-driven peak demand), monsoon (temperature relief, but
also potential grid stress from storms), and winter (mild heating demand,
generally the easiest season to forecast). Grouping by month into these
three buckets tests whether either model has a specific weak season.


In [ ]:
def month_to_season(m):
    if m in [4, 5, 6]:
        return "Summer (Apr-Jun)"
    elif m in [7, 8, 9]:
        return "Monsoon (Jul-Sep)"
    else:
        return "Winter/Other (Oct-Mar)"

merged["season"] = merged["month"].apply(month_to_season)

seasonal_mape = merged.groupby("season").apply(
    lambda g: pd.Series({
        "SARIMA_MAPE": mape(g["y_true"], g["sarima_pred"]),
        "LightGBM_MAPE": mape(g["y_true"], g["lgb_pred"]),
        "n_hours": len(g),
    })
).reset_index()

seasonal_mape

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(seasonal_mape))
width = 0.35
ax.bar(x - width/2, seasonal_mape["SARIMA_MAPE"], width, label="SARIMA")
ax.bar(x + width/2, seasonal_mape["LightGBM_MAPE"], width, label="LightGBM")
ax.set_xticks(x)
ax.set_xticklabels(seasonal_mape["season"])
ax.set_ylabel("MAPE (%)")
ax.set_title("Seasonal Error Comparison")
ax.legend()
plt.tight_layout()
plt.show()

**What to look for:** if LightGBM's relative advantage (or disadvantage)
is concentrated in one season, that's a real, explainable finding — e.g.
if LightGBM does relatively better in summer, that would suggest its access
to temperature data pays off specifically when weather is the dominant
driver of demand swings.


## 2. Temperature-stratified error

More direct than season: bin by actual temperature and compare. This tests
the hypothesis directly rather than through the proxy of "which month."


In [ ]:
merged["temp_bin"] = pd.cut(
    merged["temperature_2m"],
    bins=[-100, 15, 22, 28, 34, 100],
    labels=["<15°C", "15-22°C", "22-28°C", "28-34°C", ">34°C"],
)

temp_mape = merged.groupby("temp_bin", observed=True).apply(
    lambda g: pd.Series({
        "SARIMA_MAPE": mape(g["y_true"], g["sarima_pred"]),
        "LightGBM_MAPE": mape(g["y_true"], g["lgb_pred"]),
        "n_hours": len(g),
    })
).reset_index()

temp_mape

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(temp_mape))
width = 0.35
ax.bar(x - width/2, temp_mape["SARIMA_MAPE"], width, label="SARIMA")
ax.bar(x + width/2, temp_mape["LightGBM_MAPE"], width, label="LightGBM")
ax.set_xticks(x)
ax.set_xticklabels(temp_mape["temp_bin"])
ax.set_ylabel("MAPE (%)")
ax.set_xlabel("Temperature bin")
ax.set_title("Error by Temperature Range")
ax.legend()
plt.tight_layout()
plt.show()

**What to look for:** if LightGBM's error grows more slowly than
SARIMA's as temperature rises into the extreme range (>34°C), that's
direct evidence that weather-awareness specifically helps during heat
extremes — exactly the kind of period where AC-driven demand becomes
hardest to predict from the load series alone. If the gap doesn't widen,
that's an equally valid (and slightly surprising) finding worth reporting.


## 3. Time-of-day error

Do either model's errors concentrate around the fast transitions (morning
ramp-up, evening peak) rather than the stable overnight trough?


In [ ]:
hourly_mape = merged.groupby("hour").apply(
    lambda g: pd.Series({
        "SARIMA_MAPE": mape(g["y_true"], g["sarima_pred"]),
        "LightGBM_MAPE": mape(g["y_true"], g["lgb_pred"]),
    })
).reset_index()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(hourly_mape["hour"], hourly_mape["SARIMA_MAPE"], marker="o", label="SARIMA")
ax.plot(hourly_mape["hour"], hourly_mape["LightGBM_MAPE"], marker="o", label="LightGBM")
ax.set_xlabel("Hour of day")
ax.set_ylabel("MAPE (%)")
ax.set_title("Error by Hour of Day")
ax.legend()
ax.set_xticks(range(0, 24, 2))
plt.tight_layout()
plt.show()

**What to look for:** errors spiking around 6-9am and 6-9pm (the fast
ramp periods) would suggest both models struggle more with rate-of-change
than with absolute level — a natural limitation of models trained on MAPE
loss, which weighs all hours equally rather than penalizing missed
transitions more heavily.


## 4. Holiday vs non-holiday error

The EDA (`01_eda.ipynb`) found the holiday effect on raw demand was noisy,
and SHAP (`03_ml_models.ipynb`) found `is_holiday` had near-zero feature
importance. This checks whether that translates into a real forecasting
accuracy difference, or whether holidays are simply not very informative
for either model on this dataset.


In [ ]:
holiday_mape = merged.groupby("is_holiday").apply(
    lambda g: pd.Series({
        "SARIMA_MAPE": mape(g["y_true"], g["sarima_pred"]),
        "LightGBM_MAPE": mape(g["y_true"], g["lgb_pred"]),
        "n_hours": len(g),
    })
).reset_index()
holiday_mape["is_holiday"] = holiday_mape["is_holiday"].map({0: "Non-holiday", 1: "Holiday"})
holiday_mape

## 5. Head-to-head: which model wins more often?

Beyond averages — for what fraction of individual hours was each model
simply closer to the truth?


In [ ]:
merged["sarima_abs_err"] = (merged["y_true"] - merged["sarima_pred"]).abs()
merged["lgb_abs_err"] = (merged["y_true"] - merged["lgb_pred"]).abs()
merged["sarima_wins"] = merged["sarima_abs_err"] < merged["lgb_abs_err"]

win_rate = merged["sarima_wins"].mean()
print(f"SARIMA was closer to the truth in {win_rate:.1%} of hours")
print(f"LightGBM was closer to the truth in {1-win_rate:.1%} of hours")

# Does the win rate itself shift by season? A model can "win on average"
# while still losing more often than it wins in a specific regime.
season_win_rate = merged.groupby("season")["sarima_wins"].mean()
print("\nSARIMA win rate by season:")
print(season_win_rate)

## Summary

_(Fill in after running with your actual numbers:)_

- Which season showed the biggest gap between SARIMA and LightGBM, and in
  which direction?
- Did LightGBM's relative advantage concentrate in high-temperature
  conditions, supporting the hypothesis that weather-awareness pays off
  specifically during heat extremes? Or was the pattern flatter than
  expected?
- Where do errors spike by hour of day — transitions or troughs?
- Does the "SARIMA wins on average" headline hold up when you look at
  win-rate by season, or does LightGBM actually win more often in a
  specific regime even while losing on the overall average? (This is worth
  digging into explicitly — an aggregate average can hide a model that's
  actually more reliable in the conditions that matter most operationally,
  e.g. summer heat waves, even if it loses on a full-year average.)

## Note on scope

This analysis intentionally does **not** introduce a new model. Per the
project's own honesty principle (see README's "Is This Novel?" section):
the goal here was to extract more insight from the two models already
built and validated, rather than adding complexity for its own sake. A
feature-ablation study (removing weather/holiday features from LightGBM
one at a time and re-measuring MAPE) would be a natural next step if more
time were available, but the temperature-stratified analysis above already
gives a reasonable proxy answer to "does weather data help, and when."
